In [1]:
import polars as pl

In [2]:
event_lf = pl.scan_parquet('data/event_full_2025.parquet')
items_lf = pl.scan_parquet('data/items.parquet')
transaction_lf = pl.scan_parquet('data/transaction_full_2025.parquet')

In [3]:
display(transaction_lf.head(5).collect())
display(event_lf.head(5).collect())

display(items_lf.head(5).collect())

customer_id,item_id,price,location,discount,bill_id,quantity,event_type,updated_date
i32,str,"decimal[38,4]",i32,"decimal[38,4]",i32,i32,str,datetime[μs]
6291331,"""6767000000002""",285000.0000,1053,0.0000,137262467,1,"""Purchase""",2025-04-12 12:52:41.910
5505362,"""2265000000027""",195000.0000,468,20000.0000,138193104,1,"""Purchase""",2025-04-24 14:57:00.773
7694873,"""6497000000004""",312550.0000,363,16450.0000,138230425,1,"""Purchase""",2025-04-24 20:35:40.027
5837076,"""6587000000001""",149000.0000,48,16000.0000,138235471,1,"""Purchase""",2025-04-24 21:12:32.070
6781756,"""3880000000002""",79000.0000,634,0.0000,138239415,1,"""Purchase""",2025-04-25 08:00:46.293


customer_id,item_id,price,quantity,event_type,event_date,created_date,updated_date
i32,str,"decimal[38,4]",i32,str,datetime[μs],datetime[μs],datetime[μs]
5390942,"""2265000000025""",325000.0000,1,"""view_item""",2024-09-06 00:00:00,2025-06-04 22:37:05.605091,2025-06-04 22:37:05.605091
2525048,"""2470000000006""",555000.0000,2,"""add_to_cart""",2024-09-06 00:00:00,2025-06-04 22:37:05.605091,2025-06-04 22:37:05.605091
4243782,"""0930030380001""",19000.0000,8,"""view_item""",2024-09-06 00:00:00,2025-06-04 22:37:05.605091,2025-06-04 22:37:05.605091
6822823,"""6792000000002""",206400.0000,1,"""add_to_cart""",2024-09-06 00:00:00,2025-06-04 22:37:05.605091,2025-06-04 22:37:05.605091
1878879,"""4512000000002""",295000.0000,1,"""view_item""",2024-09-06 00:00:00,2025-06-04 22:37:05.605091,2025-06-04 22:37:05.605091


item_id,price,category_l1,category_l2,category_l3,category,brand,manufacturer,description,sale_status,size
str,"decimal[38,4]",str,str,str,str,str,str,str,i32,str
"""0008040000046""",115000.0000,"""Đồ chơi & Sách""","""1Y+""","""Học tập và phát triển tư duy""","""Siêu nhân, robot""","""WinWinToys""","""Không xác định""","""Robo Luồn thun Winwintoys có h…",0,"""Không xác định"""
"""0502020000004""",99000.0000,"""Babycare""","""Bình sữa, phụ kiện""","""Núm ty""","""Núm ty Dr Brown""","""Dr.Brown's""","""Không xác định""","""Không xác định""",0,"""Không xác định"""
"""0007010000886""",102000.0000,"""Babycare""","""Bình sữa, phụ kiện""","""Núm ty""","""Núm ty Pigeon""","""Pigeon""","""Không xác định""","""Với hơn 60 năm qua, trải qua r…",0,"""Không xác định"""
"""0502020340030""",228000.0000,"""Babycare""","""Bình sữa, phụ kiện""","""Phụ kiện bình sữa""","""Bình, túi ủ sữa""","""KUKU""","""Không xác định""","""Bình ủ sữa của KuKu của Đài Lo…",0,"""Không xác định"""
"""0012050000007""",59000.0000,"""Thời trang""","""Cơ cấu hàng cũ""","""Đồ bầu""","""Áo ngực bầu""","""Không xác định""","""Không xác định""","""Không xác định""",0,"""Không xác định"""


In [65]:
report_transaction_lf = transaction_lf.select(
    pl.col('price').min().alias('min_price'),
    pl.col('price').max().alias('max_price'),
    pl.col('discount').min().alias('min_discount'),
    pl.col('discount').max().alias('max_discount'),
    pl.col('quantity').min().alias('min_quantity'),
    pl.col('quantity').max().alias('max_quantity'),
    pl.col('updated_date').min().alias('min_updated_date'),
    pl.col('updated_date').max().alias('max_updated_date')
)

display(report_transaction_lf.collect(engine = "streaming"))

min_price,max_price,min_discount,max_discount,min_quantity,max_quantity,min_updated_date,max_updated_date
"decimal[38,4]","decimal[38,4]","decimal[38,4]","decimal[38,4]",i32,i32,datetime[μs],datetime[μs]
0.0001,20990000.0000,0.0000,13200000.0000,1,1400,2025-01-01 06:51:05.690,2025-12-31 22:30:30.267


In [66]:
report_event_lf = event_lf.select(
    pl.col('price').min().alias('min_price'),
    pl.col('price').max().alias('max_price'),
    pl.col('quantity').min().alias('min_quantity'),
    pl.col('quantity').max().alias('max_quantity'),
    pl.col('event_date').min().alias('min_date'),
    pl.col('event_date').max().alias('max_date')
)

display(report_event_lf.collect(engine = "streaming"))

min_price,max_price,min_quantity,max_quantity,min_date,max_date
"decimal[38,4]","decimal[38,4]",i32,i32,datetime[μs],datetime[μs]
-9201000.0000,20990000.0000,1,37958,2024-05-31 00:00:00,2025-12-04 00:00:00


In [67]:
### số lượng dòng có created_date == updated_date
display(event_lf.select((pl.col('created_date') != pl.col('updated_date')).sum().alias('total')).collect(engine = "streaming"))

total
u32
0


### Lọc dữ liệu 
1. Bảng event
- Lọc các giá trị âm ở cột `price`
- Bỏ đi các cột `created_date` và `updated_date`
- Thêm các cột để đồng nhất với bảng `transaction`

In [85]:
new_event_lf = (event_lf.filter(pl.col('price') >=0)
                .drop(['created_date', 'updated_date'])
                .rename({'event_date': 'updated_date'})
                .with_columns(
                    pl.lit(None).cast(pl.Int32).alias('location'),
                    pl.lit(None).cast(pl.Int32).alias('bill_id'),
                    pl.lit(None).cast(pl.Float32).alias('discount'),
                    pl.col('price').cast(pl.Float32)
                ).select([
                    'customer_id',
                    'item_id',
                    'price',
                    'location',
                    'discount',
                    'bill_id',
                    'quantity',
                    'event_type',
                    'updated_date'
                ])
)

display(new_event_lf.head(5).collect(engine='streaming'))

customer_id,item_id,price,location,discount,bill_id,quantity,event_type,updated_date
i32,str,f32,i32,f32,i32,i32,str,datetime[μs]
5390942,"""2265000000025""",325000.0,null,null,null,1,"""view_item""",2024-09-06 00:00:00
2525048,"""2470000000006""",555000.0,null,null,null,2,"""add_to_cart""",2024-09-06 00:00:00
4243782,"""0930030380001""",19000.0,null,null,null,8,"""view_item""",2024-09-06 00:00:00
6822823,"""6792000000002""",206400.0,null,null,null,1,"""add_to_cart""",2024-09-06 00:00:00
1878879,"""4512000000002""",295000.0,null,null,null,1,"""view_item""",2024-09-06 00:00:00


In [86]:
new_transaction_lf = (
    transaction_lf.with_columns(
        pl.col('price').cast(pl.Float32),
        pl.col('discount').cast(pl.Float32),
    )
)

display(new_transaction_lf.head(5).collect(engine = 'streaming'))

customer_id,item_id,price,location,discount,bill_id,quantity,event_type,updated_date
i32,str,f32,i32,f32,i32,i32,str,datetime[μs]
6291331,"""6767000000002""",285000.0,1053,0.0,137262467,1,"""Purchase""",2025-04-12 12:52:41.910
5505362,"""2265000000027""",195000.0,468,20000.0,138193104,1,"""Purchase""",2025-04-24 14:57:00.773
7694873,"""6497000000004""",312550.0,363,16450.0,138230425,1,"""Purchase""",2025-04-24 20:35:40.027
5837076,"""6587000000001""",149000.0,48,16000.0,138235471,1,"""Purchase""",2025-04-24 21:12:32.070
6781756,"""3880000000002""",79000.0,634,0.0,138239415,1,"""Purchase""",2025-04-25 08:00:46.293


In [ ]:
interaction_lf = pl.concat([new_transaction_lf, new_event_lf])

display(interaction_lf.head(5).collect(engine = 'streaming'))

customer_id,item_id,price,location,discount,bill_id,quantity,event_type,updated_date
i32,str,f32,i32,f32,i32,i32,str,datetime[μs]
104642,"""3856000000001""",375000.0,207,0.0,138240676,3,"""Purchase""",2025-04-25 08:02:02.483
4025147,"""4690000000001""",35000.0,144,0.0,138242423,3,"""Purchase""",2025-04-26 16:53:27.863
1776230,"""6768000000004""",345000.0,252,0.0,138241281,3,"""Purchase""",2025-04-25 19:17:25.047
6305030,"""6768000000003""",345000.0,212,0.0,137156835,3,"""Purchase""",2025-04-10 20:22:52.120
4283768,"""0006020000282""",25000.0,199,0.0,138242817,3,"""Purchase""",2025-04-25 08:52:41.413


### Data splitting
- Data train: trước 1/11/2025
- Data valid: trong tháng 11/2025
- Data test: trong tháng 12/2025

In [96]:
from datetime import datetime

VALID_START = datetime(2025,11,1)
TEST_START = datetime(2025,12,1)
TEST_END = datetime(2025,12,31)

In [104]:
train_set = interaction_lf.filter(pl.col('updated_date') < pl.lit(VALID_START))
valid_set = interaction_lf.filter((pl.col('updated_date') >= pl.lit(VALID_START)) & (pl.col('updated_date') < pl.lit(TEST_START)))
test_set = interaction_lf.filter((pl.col('updated_date') >= pl.lit(TEST_START)) & (pl.col('updated_date')<=pl.lit(TEST_END)))

In [108]:
display(train_set.head(5).collect(engine = 'streaming'))
display(valid_set.head(5).collect(engine = 'streaming'))
display(test_set.head(5).collect(engine = 'streaming'))

customer_id,item_id,price,location,discount,bill_id,quantity,event_type,updated_date
i32,str,f32,i32,f32,i32,i32,str,datetime[μs]
6291331,"""6767000000002""",285000.0,1053,0.0,137262467,1,"""Purchase""",2025-04-12 12:52:41.910
5505362,"""2265000000027""",195000.0,468,20000.0,138193104,1,"""Purchase""",2025-04-24 14:57:00.773
7694873,"""6497000000004""",312550.0,363,16450.0,138230425,1,"""Purchase""",2025-04-24 20:35:40.027
5837076,"""6587000000001""",149000.0,48,16000.0,138235471,1,"""Purchase""",2025-04-24 21:12:32.070
6781756,"""3880000000002""",79000.0,634,0.0,138239415,1,"""Purchase""",2025-04-25 08:00:46.293


customer_id,item_id,price,location,discount,bill_id,quantity,event_type,updated_date
i32,str,f32,i32,f32,i32,i32,str,datetime[μs]
8316345,"""4268000000069""",301506.0,345,28494.0,148702063,1,"""Purchase""",2025-11-14 14:17:34.643
6705673,"""3522000000083""",119000.0,855,0.0,148701832,1,"""Purchase""",2025-11-09 12:15:42.103
7942869,"""0009200000049""",49400.0,766,15600.0,148696856,1,"""Purchase""",2025-11-02 17:44:17.273
8061416,"""2454000000001""",785000.0,174,0.0,148706979,1,"""Purchase""",2025-11-23 08:20:59.147
6705673,"""3525000000128""",139000.0,855,0.0,148701832,1,"""Purchase""",2025-11-09 12:15:42.103


customer_id,item_id,price,location,discount,bill_id,quantity,event_type,updated_date
i32,str,f32,i32,f32,i32,i32,str,datetime[μs]
5295052,"""2507000000003""",424080.0,761,450120.0,148661126,11,"""Purchase""",2025-12-16 11:27:02.683
7955723,"""0041000000028""",407500.0,585,25000.0,148653317,2,"""Purchase""",2025-12-16 11:25:09.480
405233,"""2155000000110""",113240.0,717,35760.0,148552869,1,"""Purchase""",2025-12-16 11:22:45.320
3487571,"""2451000000001""",385000.0,342,0.0,148544814,1,"""Purchase""",2025-12-16 11:59:06.903
8269181,"""5856000000001""",480000.0,1138,0.0,148296545,1,"""Purchase""",2025-12-17 18:50:46.897


In [110]:
interaction_lf.sink_parquet('data/full_dataset.parquet')
train_set.sink_parquet('data/train_set.parquet')
valid_set.sink_parquet('data/valid_set.parquet')
test_set.sink_parquet('data/test_set.parquet')